# 07 · Clasificador multiclase: género acústico dominante

**Proyecto:** Spotify Music Intelligence
**Módulo 7:** Clasificador multiclase secundario
**Objetivo:** estimar un género acústico dominante entre 114 clases usando
únicamente grabaciones con una sola etiqueta, sin usar el test durante la
selección de modelo.

La predicción no reemplaza las etiquetas originales de canciones multigénero.
El split agrupado congelado (70/15/15) está en
`data/processed/splits.parquet`. Los árboles C2/C3 se compararon con
`scripts/compare_multiclass_models.py`; el modelo final aprobado es C1.

## Configuración y datos

Se cargan los datos procesados y el dataset multiclase construido por
`spotify_intelligence.classification.training.prepare_multiclass_data`.
No se modifica ningún dato.

In [1]:
import os
from pathlib import Path

import pandas as pd

from spotify_intelligence.classification.multiclass import (
    build_model,
    expand_to_full_label_space,
    load_model_parameters,
    model_classes,
    predict_proba_scores,
)
from spotify_intelligence.classification.multiclass_evaluation import (
    dominant_genre_exploratory,
    evaluate_multiclass,
)
from spotify_intelligence.classification.training import (
    feature_matrix,
    prepare_base_dataset,
    prepare_multiclass_data,
    split_map_from_dir,
    subset_dataset,
)
from spotify_intelligence.data.splits import verify_disjoint_splits

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

dataset = prepare_base_dataset("data/processed")
split_map = split_map_from_dir("data/processed")
print("n_samples:", dataset.n_samples)
print("n_labels:", dataset.n_labels)
print(
    "train:",
    len(split_map["train"]),
    "validation:",
    len(split_map["validation"]),
    "test:",
    len(split_map["test"]),
)

n_samples: 83881
n_labels: 114
train: 58716 validation: 12582 test: 12583


## Dataset de grabaciones monoetiqueta y split sin fuga

Solo se usan grabaciones con exactamente un género. Se verifica que
ningún `recording_group_id` aparece en dos conjuntos.

In [2]:
verify_disjoint_splits(split_map)
print("OK: intersecciones vacías entre train/validation/test")

data = prepare_multiclass_data(experiment="A")
print("X_train:", data.X_train.shape)
print("X_val:", data.X_val.shape)
print("X_test:", "None (solo con --use-test)", None if data.X_test is None else data.X_test.shape)

y_train = data.y_train
class_names = data.dataset.genre_encoder.classes_
counts = pd.Series(y_train).value_counts()
print("clases en train:", int(counts.shape[0]), "de", len(class_names))
print("min/max por clase:", int(counts.min()), "/", int(counts.max()))
print("media por clase:", round(float(counts.mean()), 1))

OK: intersecciones vacías entre train/validation/test


X_train: (48886, 18)
X_val: (10475, 18)
X_test: None (solo con --use-test) None
clases en train: 112 de 114
min/max por clase: 47 / 717
media por clase: 436.5


## Baseline C0 · Clase frecuente

Predice siempre la clase más frecuente de train; no usa características.

In [3]:
model_params = load_model_parameters("configs/model_parameters.yaml")
m0 = build_model("C0", model_params)
m0.fit(data.X_train, data.y_train)
dense0 = predict_proba_scores(m0, data.X_val)
full0 = expand_to_full_label_space(dense0, model_classes(m0), len(class_names))
evaluate_multiclass(data.y_val, full0, class_names)

{'accuracy': 0.012028639618138425,
 'macro_f1': 0.00021224412791246108,
 'balanced_accuracy': 0.008928571428571428,
 'top3_accuracy': 0.03408114558472554,
 'top5_accuracy': 0.04591885441527446,
 'confusion_matrix_normalized': [[0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,

## Modelo final C1 · Regresión logística

C1 se entrenó una vez con `scripts/train_multiclass_classifier.py`
(`solver=lbfgs`, `C=1.0`, `max_iter=3000`, `class_weight=balanced`). Aquí se
carga el artefacto versionado y se evalúa sobre validación. C2 (Extra Trees) y
C3 (Random Forest) alcanzaron mejor accuracy pero pesan ~650 MB cada uno;
C1 (~0,5 MB) fue aprobado como modelo final por el propietario.

In [4]:
import json

import joblib

c1_dirs = sorted(Path("models/classifier/multiclass").glob("*multiclass_C1"))
if not c1_dirs:
    raise SystemExit(
        "No se encontró el artefacto C1. Ejecute scripts/train_multiclass_classifier.py --model C1"
    )
artifact_dir = c1_dirs[-1]
manifest = json.loads((artifact_dir / "manifest.json").read_text(encoding="utf-8"))
model = joblib.load(artifact_dir / "model.joblib")
scaler = joblib.load(artifact_dir / "scaler.joblib")
print("artefacto:", artifact_dir.name)
print("split_sha256:", manifest["split_sha256"][:12] + "...")

dense = predict_proba_scores(model, data.X_val)
full = expand_to_full_label_space(dense, model_classes(model), len(class_names))
metrics = evaluate_multiclass(data.y_val, full, class_names)
{k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)}

artefacto: 20260805-1702_multiclass_C1
split_sha256: 7cdb3f42c405...


{'accuracy': 0.2234,
 'macro_f1': 0.1622,
 'balanced_accuracy': 0.1818,
 'top3_accuracy': 0.3968,
 'top5_accuracy': 0.4897}

## Evaluación exploratoria sobre grabaciones multigénero

C1 estima un único género dominante. Se evalúa con `Hit@1`, `Hit@3` y
`Recall@5` sobre filas de validación que tienen más de una etiqueta original.

In [5]:
val = subset_dataset(dataset, split_map["validation"], experiment="A")
mask = val.Y.sum(axis=1) > 1
print("filas multigénero en validación:", int(mask.sum()))

X_mg = scaler.transform(feature_matrix(val, "A")[mask])
Y_mg = val.Y[mask]
dense_mg = predict_proba_scores(model, X_mg)
full_mg = expand_to_full_label_space(dense_mg, model_classes(model), len(class_names))
dominant_genre_exploratory(Y_mg, full_mg, class_names)

filas multigénero en validación: 2086


{'hit_at_1': 0.19463087248322147,
 'hit_at_3': 0.39213806327900286,
 'recall_at_5': 0.28363563882798487,
 'rows': 2086}

## Limitaciones

- Es un laboratorio experimental; la predicción **no** reemplaza las
  etiquetas originales de canciones multigénero.
- El test congelado se usa solo en la evaluación final autorizada
  (`scripts/evaluate_final_multiclass_model.py --use-test`).
- Las puntuaciones no calibradas **no** se llaman probabilidades.
- La muestra es balanceada por bloque (1.000 filas por género); no permite
  inferir prevalencia real de Spotify.